# Generate a regular lat-lon grid with associated grid spacing metrics

In [1]:
from xgcm import Grid
import xarray as xr
import numpy as np
import utils.geo as geo

In [2]:
dlon,dlat = 1,1
grid = xr.Dataset()
lons = dlon/2 + np.arange(0,360,dlon)
lons_left = np.arange(0,360,dlon)
lats = dlat/2 -90 + np.arange(0,180,dlat)
lats_left = -90 + np.arange(0,180,dlat)
ds = xr.Dataset(
    coords={
        "x_c":(
            ["x_c"],
            lons
        ),
        "x_g":(
            ["x_g"],
            lons_left
        ),
        "y_c":(
            ["y_c"],
            lats
        ),
        "y_g":(
            ["y_g"],
            lats_left
        ),
        }
        )

In [3]:
coords = {
    'X':{'center':'x_c','left':'x_g'},
    'Y':{'center':'y_c','left':'y_g'}
}
periodic=["X"]

xgrid = Grid(ds,coords=coords,periodic=periodic)


In [4]:
ds['dxG'], ds['dyG'] = geo._degrees_to_meters(
    xgrid.diff(ds['x_c'], 'X', boundary="fill", fill_value=ds['x_c'][0]-dlon),
    xgrid.diff(ds['y_c'], 'Y', boundary="fill", fill_value=ds['y_c'][0]-dlat),
    ds['x_g'],
    ds['y_g']
    )

ds['dxC'], ds['dyC'] = geo._degrees_to_meters(
    xgrid.diff(ds['x_g'], 'X', boundary="fill", fill_value=ds['x_g'][-1]+dlon),
    xgrid.diff(ds['y_g'], 'Y', boundary="fill", fill_value=ds['y_g'][-1]+dlat),
    ds['x_c'],
    ds['y_c']
    )

ds['rC'] = ds['dxC']*ds['dyC']

/Users/gam24/.local/share/mamba/envs/core/lib/python3.11/site-packages/xgcm/grid_ufunc.py:832: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  out_dim: grid._ds.dims[out_dim] for arg in out_core_dims for out_dim in arg


In [5]:
# Regenerate grid
metrics = {
    'X':['dxC','dxG'],
    'Y':['dyC','dyG'],
    ('X','Y'):['rC']
}
xgrid = Grid(ds,coords=coords,metrics=metrics,periodic=periodic)